In [26]:
import os
import pandas as pd
import polars as pl
from xlsxwriter import Workbook
from datetime import datetime, timedelta

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

from getpass import getpass
import traceback # permet d'afficher les messages d'erreur du try except

import warnings
warnings.filterwarnings('ignore')

# Defining the connection string
driver = "SQL Server"
server = "172.16.22.227"
user = "usrSupplyChain"
pwd =  getpass()
BDD_name = "SupplyChain"

# conn = pyodbc.connect(f'DRIVER={driver}; Server={server}; UID={user}; PWD={pwd}; DataBase={BDD_name}')
connection_string = f'DRIVER={driver}; Server={server}; UID={user}; PWD={pwd}; DataBase={BDD_name}'
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)

con = engine.connect()

········


pour REACT, pas de suivi car les commandes d'achat n'apparaissent pas toutes dans le PDG (ex : 2053758 et 2053759 ont été receptionnée mais ne sont pas dans le PDG et donc ne sont pas débloquées....)

1 - fichier généré automatiquement tous les jours pour récupérer les réceptions et les expéditions  

2 - pour les déblocages, il faudra utiliser le fichier renseigné manuellement par les pilotes de flux  

3 - pour les kpi, en fonction de la centra (si c'est avec ou sans deblocage), on prendra uniquement le fichier des récep/expé (si sans déblocage) ou recep/expe + deblocage (si c'est une centra avec déblocage)


# INPUT

- ID Catalogue  
- Recap des déblocages pour les centra sur lesquelles on réalise des déblocages

## Comment identifier les commandes annulées ?

## Comment identifier les quantités confirmées ?

## Ajouter converse

## Calculer délai de livraison ✔

que pour les centra où il y a des déblocages

## Comment ajouter les déblocages du jour ? ✔

dans le Power BI, se connecter au fichier d'enregistrement des déblocages et ajouter les nouvelles lignes 

## Nécéssité d'avoir la valo débloquée du jour ✔

enregistrer la table des prix d'achat dans chaque fichier

## Ajouter un glossaire expliquant la data derrière chaque colonne ✔

## Ajouter le récap des catalogues pris en compte - des paramètres... ✔

## Ajouter un onglet d'alerte
### Doublons de déblocage ✔

## IDCatalogue

In [2]:
param = pd.read_excel(r"\\BUREAU2022\ACHATSChaussures\LOGISTIQUE\Rapports BDD Supply Chain\data_prep\SuiviCentra\Parametres_centra_kpi.xlsx",
                      sheet_name="IDCatalogue")
param

,Saison,Marque,IDCatalogue,DossierSuiviDesDeblocages,Fichier,Onglet,OngletAnnulations
0,SU 26,NIKE,2910,\\BUREAU2022\ACHATSChaussures\LOGISTIQUE\Appro...,suivi_deblocage_centra_nike-SU26.xlsx,Deblocages,Annulations
1,SU 26,NIKE,2911,NaN,NaN,NaN,NaN
2,SU 26,NIKE,2912,NaN,NaN,NaN,NaN
3,SU 26,NIKE,2913,NaN,NaN,NaN,NaN
4,SU 26,NIKE,2914,NaN,NaN,NaN,NaN
5,SU 26,NIKE,2927,NaN,NaN,NaN,NaN
6,SU 26,NIKE,2928,NaN,NaN,NaN,NaN
7,SU 26,NIKE,2930,NaN,NaN,NaN,NaN
8,SU 26,NIKE,2931,NaN,NaN,NaN,NaN
9,SU 26,NIKE,2932,NaN,NaN,NaN,NaN


# dossiers enregistrement

In [3]:
save_folder = r"\\BUREAU2022\ACHATSChaussures\LOGISTIQUE\Rapports BDD Supply Chain\data_prep\SuiviCentra"

# Déblocages

nécessaire pour réliser la courbe des déblocages : 
- date déblocage,
- qté débloquée,
- IDCommandeAchat/RefFourCouleur (optionel mais utile pour une question de traçabilité des données)
- valo débloquée (optionnel si IDCommandeAchat/RefFourCouleur car obtenu par calcul grâce à la RefFourCouleur et IDCommandeAchat)  


In [4]:
def get_deblocage(marque, saison, prix_achat):
    data = param[(param["Marque"] == marque) & (param["Saison"] == saison)].reset_index(drop=True)
    deblocage_folder = data["DossierSuiviDesDeblocages"].iloc[0]
    deblocage_file = data["Fichier"].iloc[0]
    deblocage_sheet = data["Onglet"].iloc[0]

    #if not pd.isna(deblocage_folder) and not pd.isna(deblocage_file) and not pd.isna(deblocage_sheet):
    try:
        deblocage_path = os.path.join(deblocage_folder, deblocage_file)
        print("fichier déblocage :",deblocage_path)
        df = pd.read_excel(deblocage_path, sheet_name=deblocage_sheet)
        
        # type des données
        df["DateDeblocage"] = pd.to_datetime(df["DateDeblocage"]).dt.date
        df["IDCommande"] = df["IDCommande"].astype(str)
        
        # ajout de la valo débloquée
        #prix_achat = get_PrixAchat_RefFourCoul(achat, RefFourCouleur)
        df = pd.merge(df, prix_achat,
                      left_on=["IDCommande", "ReferenceFournisseurCouleur"],
                     right_on=["IDCommandeAchat", "ReferenceFournisseurCouleur"],
                     how="left")
        df["MontantDeblocage"] = df["QuantiteDeblocagePiece"] * df["PrixAchatPiece"]
        
    #else:
    except Exception as e:
        df = pd.DataFrame()
        print("Nom de dossier, fichier et/ou onglet manquant.")
            
    return df


def deblocage_cum_sum(deblocage):
    
    if not deblocage.empty :
        deblocage = deblocage.groupby("DateDeblocage")[["QuantiteDeblocagePiece","MontantDeblocage"]].sum().reset_index()
        deblocage["QuantiteDeblocagePiece_cum"] = deblocage["QuantiteDeblocagePiece"].cumsum()
        deblocage["MontantDeblocage_cum"] = deblocage["MontantDeblocage"].cumsum()
    
    return deblocage


def get_annulations(marque, saison, prix_achat):
    """ quantité et montant annulés par la marque à ajouter au tableau de synthèse"""
    try :
        data = param[(param["Marque"] == marque) & (param["Saison"] == saison)].reset_index(drop=True)
        folder = data["DossierSuiviDesDeblocages"].iloc[0]
        file = data["Fichier"].iloc[0]
        sheet = data["OngletAnnulations"].iloc[0]

        df = pd.read_excel(os.path.join(folder,file), sheet_name=sheet)
        df["IDCommande"] = df["IDCommande"].astype(str)
        df = pd.merge(df, prix_achat,
                      left_on=["IDCommande", "ReferenceFournisseurCouleur"],
                      right_on=["IDCommandeAchat", "ReferenceFournisseurCouleur"],
                      how="left"
                     )
        df["MontantAnnulation"] = df["QuantiteAnnulationPiece"] * df["PrixAchatPiece"]
    
    except Exception as e:
        df = pd.DataFrame()
        print(e)
    
    return df
    

# Réceptions

In [5]:
def int_to_sql(liste_int):

    return "(" + ",".join([str(x) for x in liste_int]) + ")"


def str_to_sql(liste_str):

    return "('" + "','".join(liste_str) + "')"


def get_achat(idcat_sql):

    q = f"""
    SELECT 
    ca.IDCommandeAchat
    , ca.IDCatalogue
    , ca.IDArticle
    , DateLivraisonAttendue
    , (ca.QuantiteCommandee * a.QteParPack) as QuantiteCommandeePiece
    , (ca.PrixAchat / a.QteParPack) as PrixAchatPiece
    , (ca.QuantiteCommandee * ca.PrixAchat) as MontantAchat 
    FROM CommandeAchat as ca
    LEFT JOIN Article as a ON a.IDArticle = ca.IDArticle
    WHERE IDCatalogue IN {idcat_sql}
    AND Statut <> 99
    
    """

    return pd.read_sql(q, con)


def get_idcmd_achat(achat):

    return achat["IDCommandeAchat"].unique()


def get_idcmd_achat_id_catalogue(achat):

    return achat[["IDCommandeAchat", "IDCatalogue"]].drop_duplicates()


def get_reception(liste_idcmd_achat_sql):

    q = f"""  
    SELECT
    m.IDCommande
    , m.IDArticle
    , m.DateMouvement as DateReception
    , (m.QuantiteMouvement * a.QteParPack) as QuantiteReceptionPiece
    , (m.QuantiteMouvement * m.CAMVMvt) as MontantReception
    FROM Mouvement as m
    LEFT JOIN Article as a ON a.IDArticle = m.IDArticle
    WHERE IDTypeMouvement = 25
    AND IDCommande IN {liste_idcmd_achat_sql}
    """
    df = pd.read_sql(q, con)
    # transformer datetime en date
    if not df.empty:
        df["DateReception"] = pd.to_datetime(df["DateReception"])
        df["DateReception"] = df["DateReception"].dt.date
        
        # grouper par date de réception
        df = (df
          .groupby(["IDCommande", "IDArticle", "DateReception"])[["QuantiteReceptionPiece", "MontantReception"]]
          .sum()
          .reset_index()
         )
    
    return df


def recep_cum_sum(recep):
    
    if not recep.empty:
        # grouper par date de réception
        recep = recep.groupby("DateReception")[["QuantiteReceptionPiece", "MontantReception"]].sum().reset_index()
        # calcul des cumul

        recep["QuantiteReceptionPiece_cum"] = recep["QuantiteReceptionPiece"].cumsum()
        recep["MontantReception_cum"] = recep["MontantReception"].cumsum()

    return recep

def ref_four_couleur():
    q = """
    WITH ArtRefCouleur as (
    -- Requete qui renvoi le code article, unite produit, ref fournisseur et couleur (pour les pièces)    
        SELECT IDArticle, Conditionnement, CategorieFEDAS, UniteProduit, ReferenceFournisseur, CodeCouleur, QteParPack
        FROM Article
        LEFT JOIN Couleur ON IDArticle = CodeM3
        WHERE IDArticle NOT IN ('AJUSTE', 'PERTE', 'TVA20', 'TVA55', 'ZGESCOM', 'ZGESPORT', 'ZPUB', 'ZTRADE')
    ),

    ArticleReferenceFournisseur as (
    -- requete qui démultiplie les lignes de pack et renvoi la ref fournisseur de chaque code article contenu dans les pack
        SELECT a.IDArticle,
                a.CategorieFEDAS,
                a.ReferenceFournisseur,
                a.UniteProduit,
                a.QteParPack,
                ISNULL(a.CodeCouleur, '') as CodeCouleur,
                ap.CodeArticle,
                a2.ReferenceFournisseur as RefFourPiece,
                a2.CodeCouleur as CodeCouleurPiece,
        CASE
            WHEN a.UniteProduit = 'PIECE' THEN a.ReferenceFournisseur
            ELSE a2.ReferenceFournisseur
        END AS ReferenceFournisseurClean,

        CASE
            WHEN a.UniteProduit = 'PIECE' THEN ISNULL(a.CodeCouleur, '')
            ELSE ISNULL(a2.CodeCouleur, '')
        END AS CodeCouleurClean   

        FROM ArtRefCouleur AS a
        LEFT JOIN ArticlePack as ap ON a.IDArticle = ap.CodePack
        LEFT JOIN ArtRefCouleur AS a2 ON ap.CodeArticle = a2.IDArticle
        ),

    ArticleDistinct AS (
        SELECT DISTINCT IDArticle, UniteProduit, CategorieFEDAS, QteParPack,
                ReferenceFournisseurClean as ReferenceFournisseur,
                CodeCouleurClean as CodeCouleur
        FROM ArticleReferenceFournisseur
    ),

    last as (
        SELECT IDArticle, UniteProduit, ReferenceFournisseur,
                STRING_AGG(CodeCouleur, '') as CodeCouleur, QteParPack
        FROM ArticleDistinct
        GROUP BY IDArticle, UniteProduit, CategorieFEDAS,
                    ReferenceFournisseur, QteParPack
                )
    SELECT *,
    CASE
         WHEN CodeCouleur IS NOT NULL AND CodeCouleur <> '' THEN ReferenceFournisseur+'-'+CodeCouleur
         ELSE ReferenceFournisseur
    END AS ReferenceFournisseurCouleur

    FROM last

    """ 
    return pd.read_sql(q, con)

def get_PrixAchat_RefFourCoul(achat, RefFourCouleur):
    """
    renvoi le prix d'achat par ReferenceFournisseurCouleur 
    """
       
    px_achat_u = pd.merge(achat[["IDCommandeAchat", "IDArticle", "PrixAchatPiece"]], 
                      RefFourCouleur[["IDArticle", "ReferenceFournisseurCouleur"]], 
                      on="IDArticle", 
                      how='left'
                     )
    del px_achat_u["IDArticle"]
    px_achat_u = px_achat_u.drop_duplicates(subset=["IDCommandeAchat", "ReferenceFournisseurCouleur", "PrixAchatPiece"])
    px_achat_u = px_achat_u[["IDCommandeAchat", "ReferenceFournisseurCouleur", "PrixAchatPiece"]]
    
    # Y a t_il des doublons de prix d'achat à la refFourCouleur
    print("Nombre de doublons de prix :",
          px_achat_u[["ReferenceFournisseurCouleur", "PrixAchatPiece"]].drop_duplicates().duplicated().sum())
    
    return px_achat_u




# Expéditions

In [6]:
def get_vente(idcat_sql):
    """
    récupération des IDCommandeVente grace aux IDCatalogue
    puis des éxpéditions vers les magasins via les IDCommandeVente
    """
    
    q = f"""
    SELECT IDCommandeVente
    , IDCatalogue
    , IDArticle
    , SUM(QuantiteCommandee) AS QuantiteCommandee
    FROM CommandeVente
    WHERE IDCatalogue IN {idcat_sql}
    AND Statut <> 99
    GROUP BY IDCommandeVente
    , IDCatalogue
    , IDArticle
    """
    
    return pd.read_sql(q, con)

def get_idcmd_vente(vente):

    return vente["IDCommandeVente"].unique()

def get_expedition(liste_idcmd_vente_sql):
    """
    renvoi les expéditions des commandes de vente
    basée sur les IDCommandeVente
    """
    
    q = f"""
    SELECT m.IDCommande
    , m.IDArticle
    , m.DateMouvement as DateExpedition
    , (- m.QuantiteMouvement * a.QteParPack) as QuantiteExpeditionPiece
    , (- m.QuantiteMouvement * m.CAMVMvt) as MontantExpedition
    FROM Mouvement as m
    LEFT JOIN Article as a ON a.IDArticle = m.IDArticle
    WHERE IDTypeMouvement = 31
    AND IDCommande IN {liste_idcmd_vente_sql}
    """
    df = pd.read_sql(q, con)
    # transformer datetime en date
    if not df.empty:
        df["DateExpedition"] = pd.to_datetime(df["DateExpedition"])
        df["DateExpedition"] = df["DateExpedition"].dt.date
        
    return df

def expe_cum_sum(expedition):
    
    if not expedition.empty:

        # grouper par date de réception
        expedition = expedition.groupby("DateExpedition")[["QuantiteExpeditionPiece", "MontantExpedition"]].sum().reset_index()
        # calcul des cumul

        expedition["QuantiteExpeditionPiece_cum"] = expedition["QuantiteExpeditionPiece"].cumsum()
        expedition["MontantExpedition_cum"] = expedition["MontantExpedition"].cumsum()

    return expedition


# Calculs - Enregistrements

In [57]:
def calcul_cumsum(df):
    # colonne non date
    other_col = [col for col in df.columns if not col.startswith("Date")]
    for col in other_col:
        new_col = col + "_cum"
        df[new_col] = df[col].cumsum()

    return df


def calcul_pct(df, col_num, denom, new_col_name):

    if not df.empty:
        df[new_col_name] = df[col_num] / denom

    return df


def calcul_pct_df(df, denom_qte, denom_valo):

    if not df.empty:
        # colonnes quantité cumulé à utiliser
        for col in [col for col in df.columns if col.startswith("Quantite")]:
            new_col = "pct_" + col
            df[new_col] = df[col] / denom_qte
        for col in [col for col in df.columns if col.startswith("Montant")]:
            new_col = "pct_" + col
            df[new_col] = df[col] / denom_valo

    return df


def calcul_pct_cum_df(df, denom_qte, denom_valo):

    if not df.empty:
        # colonnes quantité cumulé à utiliser
        for col in [
                col for col in df.columns
                if col.startswith("Quantite") and col.endswith("_cum")
        ]:
            new_col = "pct_" + col
            df[new_col] = df[col] / denom_qte
        for col in [
                col for col in df.columns
                if col.startswith("Montant") and col.endswith("_cum")
        ]:
            new_col = "pct_" + col
            df[new_col] = df[col] / denom_valo
    return df


def calc_synthese(achat, reception, expedition, deblocage, annulation):  # ensuite ajouter déblocage

    qte_achat = achat["QuantiteCommandeePiece"].sum()
    qte_recep = reception["QuantiteReceptionPiece"].sum()
    qte_expe = expedition["QuantiteExpeditionPiece"].sum()

    valo_achat = achat["MontantAchat"].sum()
    valo_recep = reception["MontantReception"].sum()
    valo_expe = expedition["MontantExpedition"].sum()

    pct_recep = round(qte_recep / qte_achat, 3)
    pct_expe = round(qte_expe / qte_achat, 3)

    pct_valo_recep = round(valo_recep / valo_achat, 3)
    pct_valo_expe = round(valo_expe / valo_achat, 3)

    if not deblocage.empty:
        qte_debl = deblocage["QuantiteDeblocagePiece"].sum()
        valo_debl = deblocage["MontantDeblocage"].sum()
        pct_debl = round(qte_debl / qte_achat, 3)
        pct_valo_debl = round(valo_debl / valo_achat, 3)

        data = {
            "Unité": ["QuantitePiece", "Valo €"],
            "Commande": [qte_achat, valo_achat],
            "Deblocage": [qte_debl, valo_debl],
            "Reception": [qte_recep, valo_recep],
            "Expedition": [qte_expe, valo_expe],
            "% Deblocage": [pct_debl, pct_valo_debl],
            "% Reception": [pct_recep, pct_valo_recep],
            "% Expedition": [pct_expe, pct_valo_expe]
        }
    else:
        data = {
            "Unité": ["QuantitePiece", "Valo €"],
            "Commande": [qte_achat, valo_achat],
            "Reception": [qte_recep, valo_recep],
            "Expedition": [qte_expe, valo_expe],
            "% Reception": [pct_recep, pct_valo_recep],
            "% Expedition": [pct_expe, pct_valo_expe]
        }
        
    if not annulation.empty:
        qte_annul = annulation["QuantiteAnnulationPiece"].sum()
        valo_annul = annulation["MontantAnnulation"].sum()
        pct_annul = round(qte_annul / qte_achat, 3)
        pct_valo_annul = round(valo_annul / valo_achat, 3)
        
        data["Annulation"] = [qte_annul, valo_annul]
        data["% Annulation"] = [pct_annul, pct_valo_annul]
    
    df = pd.DataFrame(data=data)
    #df.index = ["QuantitePiece", "Valo €"]
    return df


def add_idcatalogue(reception, achat, expedition, vente):

    reception = pd.merge(reception,
                         achat[["IDCatalogue",
                                "IDCommandeAchat"]].drop_duplicates(),
                         left_on="IDCommande",
                         right_on="IDCommandeAchat",
                         how="left")
    expedition = pd.merge(expedition,
                          vente[["IDCatalogue",
                                 "IDCommandeVente"]].drop_duplicates(),
                          left_on="IDCommande",
                          right_on="IDCommandeVente",
                          how="left")
    return reception, expedition


def aggregate_by_date(deblocage, reception, expedition):
    # grouper par date :
    if not deblocage.empty:
        # grouper par date de réception
        deblocage = deblocage.groupby("DateDeblocage")[[
            "QuantiteDeblocagePiece", "MontantDeblocage"
        ]].sum().reset_index()
    if not reception.empty:
        # grouper par date de réception
        reception = reception.groupby("DateReception")[[
            "QuantiteReceptionPiece", "MontantReception"
        ]].sum().reset_index()
    if not expedition.empty:
        # grouper par date de réception
        expedition = expedition.groupby("DateExpedition")[[
            "QuantiteExpeditionPiece", "MontantExpedition"
        ]].sum().reset_index()

    return deblocage, reception, expedition


def merge_df(deblocage_grouped, reception_grouped, expedition_grouped):
    """
    renvoi les données des déblocages, expédition et / ou réception compilées
    """
    # savoir quels sont les df non vides
    not_empty_df = [
        df
        for df in [deblocage_grouped, reception_grouped, expedition_grouped]
        if not df.empty
    ]

    if len(not_empty_df) == 1:
        df = not_empty_df[0]
    elif len(not_empty_df) == 2:
        # merge des 2 df sur les colonnes de date
        df1 = not_empty_df[0]
        df2 = not_empty_df[1]
        col_date1 = [col for col in df1 if col.startswith("Date")][0]
        col_date2 = [col for col in df2 if col.startswith("Date")][0]
        df = pd.merge(df1,
                      df2,
                      left_on=col_date1,
                      right_on=col_date2,
                      how="outer")
    elif len(not_empty_df) == 3:  # 3 dataframes à fusionner
        df1 = not_empty_df[0]
        df2 = not_empty_df[1]
        df3 = not_empty_df[2]
        col_date1 = [col for col in df1 if col.startswith("Date")][0]
        col_date2 = [col for col in df2 if col.startswith("Date")][0]
        col_date3 = [col for col in df3 if col.startswith("Date")][0]
        df = pd.merge(df1,
                      df2,
                      left_on=col_date1,
                      right_on=col_date2,
                      how="outer")
        # colonne de date intermédiaire, utilisée pour le 2ème merge
        df["Date_int"] = df[[col_date1,
                             col_date2]].bfill(axis=1).ffill(axis=1).iloc[:, 0]
        df = pd.merge(df,
                      df3,
                      left_on="Date_int",
                      right_on=col_date3,
                      how="outer")
        df = df.drop(columns="Date_int")
    
    else:
        df = pd.DataFrame()

    return df


def sum_by_date(df):
    """
    renvoi un tableau des données de déblocage, réception, expédition en date
    """
    if not df.empty:
        # grouper par date
        # Colonne unique de date
        # on n'a pas forcément les 3 colonnes de date, on peut en avoir que 2 (si c'est une centra sans déblocage)
        # identifier les colonnes de date par leur nom
        colonnes_dates = [col for col in df.columns if col.startswith("Date")]

        # on rempli horizontalement (axis=1) avec la première date non vide (vers la gauche puis vers la droite)
        # puis on prend la 1ère colonne
        df["Date"] = df[colonnes_dates].bfill(axis=1).ffill(axis=1).iloc[:, 0]

        # grouper par Date
        other_col = [col for col in df.columns if not col.startswith("Date")]
        df1 = df.groupby("Date")[other_col].sum().reset_index()

        # calculer les cumsum
        df1 = calcul_cumsum(df1)
    else:
        df1 = pd.DataFrame()

    return df1


def sum_by_date_deblocage(df):
    """
    renvoi un tableau des données de déblocage, réception, expédition en date de déblocage
    """
    # quand il y a des NaN dans les dates de déblocages (=> il y a des réceptions ou expé mais pas de déblocage)
    # alors on rempli les NaN avec la prochaine date de déblocage
    # pour les réceptions/expéditions de fin de saison, il n'y a pas de déblocage à venir (car les déblocages sont finis)
    # donc on met la date max entre la réception et l'expédition
    if not df.empty:
        # identifier les colonnes de date
        col_date = [col for col in df.columns if col.startswith("Date")]

        if "DateDeblocage" in df.columns:
            if len(col_date) == 4:
                max_dt = max(
                    df[df["DateExpedition"].notna()]["DateExpedition"].max(),
                    df[df["DateReception"].notna()]["DateReception"].max())
            elif len(col_date) == 3:
                if "DateExpedition" in col_date:
                    max_dt = df[
                        df["DateExpedition"].notna()]["DateExpedition"].max()
                elif "DateReception" in col_date:
                    max_dt = df[df["DateReception"].notna()]["DateReception"].max()
                else:
                    max_dt = df[df["DateDeblocage"].notna()]["DateDeblocage"].max()
            else:
                max_dt = df[df["DateDeblocage"].notna()]["DateDeblocage"].max()

            df["DateDeblocage"] = df["DateDeblocage"].bfill()
            df["DateDeblocage"] = df["DateDeblocage"].fillna(max_dt)

            col_non_date = [
                col for col in df.columns if not col.startswith("Date")
            ]

            df1 = df.groupby("DateDeblocage")[col_non_date].sum().reset_index()
            # calculer les cumsum
            df1 = calcul_cumsum(df1)
        else:
            df1 = pd.DataFrame()
    else:
        df1 = pd.DataFrame()
    return df1

def calc_delai_livraison(deblocage, reception, RefFourCouleur):
    """ le délai théorique entre la livraison et la réception est de 24h pour Logs
        pondéré par la qté receptionnée    
    """
    if not deblocage.empty and not reception.empty:
        # passer les réceptions en par ref fournisseur
        reception_ref = pd.merge(reception, RefFourCouleur[["IDArticle", "ReferenceFournisseurCouleur"]], on="IDArticle", how="left")
        
#         # grouper par IDCommande/RefFourCouleur date première réception
#         reception_ref = (reception_ref
#                          .groupby(["IDCatalogue", "IDCommande", "ReferenceFournisseurCouleur"])
#                          .agg({"DateReception": min}))
        
        # lier avec les déblocages
        delai = pd.merge(deblocage, reception_ref, on=["IDCommande", "ReferenceFournisseurCouleur"], how="left")
        
        # calculer le delta entre le déblocage et la réception
        delai["delai j"] = delai["DateReception"] - delai["DateDeblocage"] + pd.Timedelta(days=-1)
        # supprimer les lignes où la date de réception est antérieure à la date de déblocage
        delai = delai[delai["delai j"] > timedelta(days=0)]
        
        delai["delai*qte"] = delai["delai j"] * delai["QuantiteReceptionPiece"]/1000 # division par 1000 car sinon renvoi une erreur le chiffre est too big
        
        
        # grouper par date de déblocage - aggreger par moyenne de délai
        delai_g = delai.groupby("DateDeblocage", dropna=False).agg({"delai j":"mean", "QuantiteReceptionPiece": sum, "delai*qte":sum}).reset_index()
        # calculer delai moyen pondéré
        delai_g["delai pondere j"] = delai_g["delai*qte"] / delai_g["QuantiteReceptionPiece"] *1000
        delai_g.drop(columns=["QuantiteReceptionPiece", "delai*qte"], inplace=True)
        
    
    else:
        delai_g = pd.DataFrame()
    
    return delai_g

def calc_retard_livraisons(achat, reception, RefFourCouleur):
    
    if not reception.empty:
        df = pd.merge(achat, reception, left_on=["IDCommandeAchat", "IDArticle"], right_on=["IDCommande", "IDArticle"], how="left")
        df["DateReception"] = pd.to_datetime(df["DateReception"])
        df["DateLivraisonAttendue"] = pd.to_datetime(df["DateLivraisonAttendue"])
        df["Reception-attendu"] = df["DateReception"] - df["DateLivraisonAttendue"] - timedelta(hours = 24)
        # SI Reception-attendu > 0 alors il ya un retard de livraison
        df["RetardLivraison"] = df["Reception-attendu"].apply(lambda x: x > timedelta(days=0))
        # calcul du retard de livraison pondéré par la quantité réceptionnée
        df["Reception*Qte/1000"] = df["Reception-attendu"] * df["QuantiteReceptionPiece"] /1000 # diviser par 1000 sinon ça fait un int too big
        ## Focus sur les retards
        retard = df[df["RetardLivraison"]]

        ## en ref fournisseur Couleur

        retard = pd.merge(retard, RefFourCouleur, on="IDArticle", how="left")
        retard_g = (retard
                    .groupby(["ReferenceFournisseurCouleur",
                              "IDCommandeAchat", "IDCatalogue",
                              "DateLivraisonAttendue","QuantiteCommandeePiece", "PrixAchatPiece", "MontantAchat"
                             ])
                    .agg(DateReceptionMin = ("DateReception", min), 
                         DateReceptionMax = ("DateReception", max),
                         MontantReception = ("MontantReception", sum),
                         ReceptionQteDiv1000 = ("Reception*Qte/1000", sum), 
                         QuantiteReceptionPiece = ("QuantiteReceptionPiece", sum)
                        )
                    .reset_index()
                   )
        retard_g["RetardMoyenPondéré"] = retard_g["ReceptionQteDiv1000"] / retard_g["QuantiteReceptionPiece"] * 1000

    else:
        retard_g=pd.DataFrame()
    
    return retard_g


def set_glossaire():

    data = {
        "Remarque": [
            "les % sont calculés en divisant par les quantités ou montants des commandes d'achat dans M3",
            "les quantités sont exprimées en pièces",
            """délai pondéré = délai de livraison pondéré par les quantités réceptionnées.\n
            Un délai de 24h est soustrait, ce qui correspond au temps laissé à l'entrepôt pour réaliser les réceptions""",
            "délai livraison = DateRéception - DateDéblocage - 24h",
            "RetardLivraison = DateReception - DateLivraisonAttendueCmdAchat - 24h"
            "RetardMoyenPondéré = retard moyen en jours pondéré par la qté réceptionnée"
        ]
    }

    return pl.DataFrame(data)


def get_warnings(deblocage):
    if not deblocage.empty:
        # doublons de déblocage
        doublons = deblocage[deblocage[[
            "IDCommande", "ReferenceFournisseurCouleur"
        ]].duplicated()][["IDCommande", "ReferenceFournisseurCouleur"]]
        df = pd.merge(deblocage, doublons, on=["IDCommande", "ReferenceFournisseurCouleur"], how="inner")
    else:
        df = pd.DataFrame()

    return df


def save(marque, saison, parametrage, synthese, reception, recep, expedition,
         expe, achat, vente, deblocage, debl, data_by_date,
         data_by_date_deblocage, prix_achat, delai_livraison, warning, retard, annulation):
    save_name = "_".join(["suivi", marque, saison]) + ".xlsx"
    with Workbook(os.path.join(save_folder, save_name)) as wb:
        glossaire = set_glossaire()
        glossaire.write_excel(workbook=wb, worksheet="glossaire", autofit=True)
        # enregistrer les paramètres utilisés
        pl.from_pandas(parametrage).write_excel(workbook=wb,
                                                worksheet="parametrage",
                                                autofit=True)
        
        # enregistrer les alertes (doublons)
        if not warning.empty:
            ws = wb.add_worksheet("alerte")
            ws.write("A1", "Doublons de déblocage") # permet d'ajouter une ligne avec du texte avant le tableau
            pl.from_pandas(warning).write_excel(workbook=wb, worksheet=ws, position="A2", autofit=True)        
        
        # enregistrer les data
        pl.from_pandas(synthese).write_excel(workbook=wb,
                                             worksheet="synthese",
                                             autofit=True)
        pl.from_pandas(data_by_date).write_excel(workbook=wb,
                                                 worksheet="par date",
                                                 autofit=True)
        if not data_by_date_deblocage.empty:
            pl.from_pandas(data_by_date_deblocage).write_excel(
                workbook=wb, worksheet="par date deblocage", autofit=True)
        if not delai_livraison.empty:
            pl.from_pandas(delai_livraison).write_excel(
                workbook=wb, worksheet="delai livraison", autofit=True)
        if not debl.empty:
            pl.from_pandas(debl).write_excel(workbook=wb,
                                             worksheet="deblocages",
                                             autofit=True)
        pl.from_pandas(recep).write_excel(workbook=wb,
                                          worksheet="reception",
                                          autofit=True)
        pl.from_pandas(expe).write_excel(workbook=wb,
                                         worksheet="expedition",
                                         autofit=True)
        if not deblocage.empty:
            pl.from_pandas(deblocage).write_excel(workbook=wb,
                                                  worksheet="detail_deblocage",
                                                  autofit=True)
        if not annulation.empty:
            pl.from_pandas(annulation).write_excel(workbook=wb,
                                                  worksheet="detail_annulation",
                                                  autofit=True)
        pl.from_pandas(reception).write_excel(workbook=wb,
                                              worksheet="detail_recep",
                                              autofit=True)
        if not retard.empty:
            pl.from_pandas(retard).write_excel(workbook=wb,
                                               worksheet="retardLivraison",
                                               autofit=True
                                              )
        pl.from_pandas(expedition).write_excel(workbook=wb,
                                               worksheet="detail_expe",
                                               autofit=True)
        pl.from_pandas(achat).write_excel(workbook=wb,
                                          worksheet="commande_achat",
                                          autofit=True)
        pl.from_pandas(vente).write_excel(workbook=wb,
                                          worksheet="commande_vente",
                                          autofit=True)
        pl.from_pandas(prix_achat).write_excel(workbook=wb,
                                               worksheet="prix_achat",
                                               autofit=True)

    return

# Calcul par référence fournisseur couleur

In [8]:
def groupby_ref_couleur_date(df, RefFourCouleur):
    if not df.empty:
        # ajouter la référence fournisseur couleur
        if "ReferenceFournisseurCouleur" not in df.columns:
            df = pd.merge(df, RefFourCouleur[["IDArticle", "ReferenceFournisseurCouleur"]], on="IDArticle", how="left")

        # grouper par ref four couleur et date
        col_gp = ["ReferenceFournisseurCouleur"] + [col for col in df.columns if col.startswith("Date")]
        col_agg = [col for col in df.columns if (col.startswith("Quantite") and col.endswith("Piece")) or col.startswith("Montant")]

        df = df.groupby(col_gp)[col_agg].sum().reset_index()
    
    return df


def get_ref_date(achat_ref, deblocage_ref_date, reception_ref_date, expedition_ref_date):
    """
    renvoi un dataframe avec toutes les referencesfournisseurs couleurs possibles et toutes les dates de mouvement
    """
    
    # récupération de toutes les réf couleur possibles pour les df non vides
    ref_all = pd.concat([
                        df["ReferenceFournisseurCouleur"]
                        for df in [achat_ref, deblocage_ref_date, reception_ref_date, expedition_ref_date]
                        if not df.empty
                        ]).drop_duplicates()

    RefFourCoul_all = pd.DataFrame({"ReferenceFournisseurCouleur":ref_all})

    # récupération de toutes les dates de mouvement
    DateMvt = pd.DataFrame()
    for df in [deblocage_ref_date, reception_ref_date, expedition_ref_date]:
        col = [col for col in df.columns if col.startswith("Date")]
        date_mvt = df[col]
        date_mvt = date_mvt.rename(columns = {col[0]: "Date"})
        DateMvt = pd.concat([DateMvt, date_mvt]).drop_duplicates()


    # récupération de toutes les dates de déblocage
    DateDebl = pd.DataFrame({"DateDeblocage":deblocage_ref_date["DateDeblocage"].drop_duplicates()})

    # produit cartésien pour avoir pour chaque ref une ligne pour chaque date
    ref_date_mvt = pd.merge(RefFourCoul_all, DateMvt, how="cross").sort_values(by=["ReferenceFournisseurCouleur", "Date"])


    return ref_date_mvt



def merge_df_by_ref_four(deblocage_grouped, reception_grouped, expedition_grouped, ref_date_mvt):
    """
    renvoi les données des déblocages, expédition et / ou réception compilées
    """
    # savoir quels sont les df non vides
    not_empty_df = [
        df
        for df in [deblocage_grouped, reception_grouped, expedition_grouped]
        if not df.empty
    ]

    if len(not_empty_df) == 1:
        df = not_empty_df[0]
        # ajouter toutes les ref four couleur présentes dans les commandes d'achat
        col_date = [col for col in df if col.startswith("Date")][0]
        df = pd.merge(ref_date_mvt,
                      df,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date],
                      how="left")

    elif len(not_empty_df) == 2:
        # merge des 2 df sur les colonnes de date
        df1 = not_empty_df[0]
        df2 = not_empty_df[1]
        col_date1 = [col for col in df1 if col.startswith("Date")][0]
        col_date2 = [col for col in df2 if col.startswith("Date")][0]
        # ajouter toutes les ref sur toutes les dates
        df = pd.merge(ref_date_mvt,
                      df1,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date1],
                      how="left"
                     )
        df = pd.merge(df,
                      df2,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date2],
                      how="left")
    elif len(not_empty_df) == 3:  # 3 dataframes à fusionner
        df1 = not_empty_df[0]
        df2 = not_empty_df[1]
        df3 = not_empty_df[2]
        col_date1 = [col for col in df1 if col.startswith("Date")][0]
        col_date2 = [col for col in df2 if col.startswith("Date")][0]
        col_date3 = [col for col in df3 if col.startswith("Date")][0]

        df = pd.merge(ref_date_mvt,
                      df1,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date1],
                      how="left"
                     )
        df = pd.merge(df,
                      df2,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date2],
                      how="left")
        df = pd.merge(df,
                      df3,
                      left_on=["ReferenceFournisseurCouleur", "Date"],
                      right_on=["ReferenceFournisseurCouleur", col_date3],
                      how="left")

    else:
        df = pd.DataFrame()

    return df


def cumsum_by_ref_date(df):
    """
    renvoi un tableau des données de déblocage, réception, expédition en date et les sommes cumulées
    """
    if not df.empty:
        # grouper par date
        # Colonne unique de date
        # on n'a pas forcément les 3 colonnes de date, on peut en avoir que 2 (si c'est une centra sans déblocage)
        # identifier les colonnes de date par leur nom
        colonnes_dates = [col for col in df.columns if col.startswith("Date")]

        # on rempli horizontalement (axis=1) avec la première date non vide (vers la gauche puis vers la droite)
        # puis on prend la 1ère colonne
        df["Date"] = df[colonnes_dates].bfill(axis=1).ffill(axis=1).iloc[:, 0]

        # grouper par Date
        other_col = [col for col in df.columns if not col.startswith("Date")]
        other_col.remove("ReferenceFournisseurCouleur")
        df1 = df.groupby(["ReferenceFournisseurCouleur", "Date"])[other_col].sum().reset_index()

        # calculer les cumsum
        df1 = calcul_cumsum_by_ref(df1)
    else:
        df1 = pd.DataFrame()

    return df1


def calcul_cumsum_by_ref(df):
    # colonne non date
    other_col = [col for col in df.columns if not col.startswith("Date") and not col.startswith("Reference")]
    for col in other_col:
        new_col = col + "_cum"
        df[new_col] = df.groupby(["ReferenceFournisseurCouleur"])[col].cumsum()

    return df


def sum_by_ref_date_deblocage(df):
    """
    renvoi un tableau des données de déblocage, réception, expédition en date de déblocage
    """
    # quand il y a des NaN dans les dates de déblocages (=> il y a des réceptions ou expé mais pas de déblocage)
    # alors on rempli les NaN avec la prochaine date de déblocage
    # pour les réceptions/expéditions de fin de saison, il n'y a pas de déblocage à venir (car les déblocages sont finis)
    # donc on met la date max entre la réception et l'expédition
    if not df.empty:
        # identifier les colonnes de date
        col_date = [col for col in df.columns if col.startswith("Date")]

        if "DateDeblocage" in df.columns:
            if len(col_date) == 4:
                max_dt = df["Date"].max() # permet de mettre une date pour les lorsque les déblocages sont finis
                                        # et qu'il reste des réceptions expéditions à venir
            df["DateDeblocage"] = df.groupby("ReferenceFournisseurCouleur")["DateDeblocage"].bfill()      

            df["DateDeblocage"] = df["DateDeblocage"].fillna(max_dt)

            col_non_date = [
                col for col in df.columns if not col.startswith("Date") and not col.startswith("Reference")
            ]
            df1 = df.groupby(["ReferenceFournisseurCouleur", "DateDeblocage"])[col_non_date].sum().reset_index()

            # récupérer les date déblocage
            dt_debl = pd.DataFrame({"DateDeblocage":df["DateDeblocage"].drop_duplicates()})
            # récupérer toutes les réf fournisseur
            ref_debl = pd.DataFrame({"ReferenceFournisseurCouleur":df["ReferenceFournisseurCouleur"].drop_duplicates()})
            # tableau de toutes les réf et toutes les dates de déblocage
            ref_dt_debl = pd.merge(ref_debl, dt_debl, how="cross").sort_values(by=["ReferenceFournisseurCouleur", "DateDeblocage"])

            df2 = pd.merge(ref_dt_debl, df1, on=["ReferenceFournisseurCouleur","DateDeblocage"], how="left")
            df2 = df2.fillna(0)
            # calculer les cumsum
            df3 = calcul_cumsum_by_ref(df2)
        else:
            df3 = pd.DataFrame()
    else:
        df3 = pd.DataFrame()
    return df3

def calc_delai_livraison_by_ref(deblocage, reception_ref_date, RefFourCouleur):
    """ le délai théorique entre la livraison et la réception est de 24h pour Logs
        pondéré par la qté receptionnée    
    """
    if not deblocage.empty and not reception.empty:
        # passer les réceptions en par ref fournisseur
        reception_ref = pd.merge(reception, RefFourCouleur[["IDArticle", "ReferenceFournisseurCouleur"]], on="IDArticle", how="left")
        # puis grouper par ref four couleur
        reception_ref = (reception_ref
                         .groupby(["IDCommande", "ReferenceFournisseurCouleur", "DateReception"])[["QuantiteReceptionPiece", "MontantReception"]]
                         .sum()
                         .reset_index()
                        )

        delai = pd.merge(deblocage, reception_ref, on=["IDCommande", "ReferenceFournisseurCouleur"], how="left")
        # supprimer les lignes ou la date de réception est antérieure à la date de déblocage
        delai = delai[delai["DateReception"] > delai["DateDeblocage"]]
        # calculer le delta entre le déblocage et la réception
        delai["delai j"] = delai["DateReception"] - delai["DateDeblocage"] + pd.Timedelta(days=-1)
        # supprimer les lignes où la date de réception est antérieure à la date de déblocage
        delai = delai[delai["delai j"] > timedelta(days=0)]
        
        delai["delai*qte"] = delai["delai j"] * delai["QuantiteReceptionPiece"]/1000 # division par 1000 car sinon renvoi une erreur le chiffre est too big
        
        
        # grouper par date de déblocage - aggreger par moyenne de délai
        delai_g = delai.groupby(["ReferenceFournisseurCouleur","DateDeblocage"], dropna=False).agg({"delai j":"mean", "QuantiteReceptionPiece": sum, "delai*qte":sum}).reset_index()
        # calculer delai moyen pondéré
        delai_g["delai pondere j"] = delai_g["delai*qte"] / delai_g["QuantiteReceptionPiece"] *1000
        delai_g.drop(columns=["QuantiteReceptionPiece", "delai*qte"], inplace=True)
        
    
    else:
        delai_g = pd.DataFrame()
    
    return delai_g


def save_by_ref(marque, saison, parametrage, synthese, reception, recep, expedition,
         expe, achat, vente, deblocage, debl, data_by_date,
         data_by_date_deblocage, prix_achat, delai_livraison, warning, retard, annulation):
    save_name = "_".join(["suivi_par_ref", marque, saison]) + ".xlsx"
    with Workbook(os.path.join(save_folder, save_name)) as wb:
        glossaire = set_glossaire()
        glossaire.write_excel(workbook=wb, worksheet="glossaire", autofit=True)
        # enregistrer les paramètres utilisés
        pl.from_pandas(parametrage).write_excel(workbook=wb,
                                                worksheet="parametrage",
                                                autofit=True)
        
        # enregistrer les alertes (doublons)
        if not warning.empty:
            ws = wb.add_worksheet("alerte")
            ws.write("A1", "Doublons de déblocage") # permet d'ajouter une ligne avec du texte avant le tableau
            pl.from_pandas(warning).write_excel(workbook=wb, worksheet=ws, position="A2", autofit=True)        
        
        # enregistrer les data
        pl.from_pandas(synthese).write_excel(workbook=wb,
                                             worksheet="synthese",
                                             autofit=True)
        pl.from_pandas(data_by_date).write_excel(workbook=wb,
                                                 worksheet="par date",
                                                 autofit=True)
        if not data_by_date_deblocage.empty:
            pl.from_pandas(data_by_date_deblocage).write_excel(
                workbook=wb, worksheet="par date deblocage", autofit=True)
        if not delai_livraison.empty:
            pl.from_pandas(delai_livraison).write_excel(
                workbook=wb, worksheet="delai livraison", autofit=True)
        if not debl.empty:
            pl.from_pandas(debl).write_excel(workbook=wb,
                                             worksheet="deblocages",
                                             autofit=True)
        pl.from_pandas(recep).write_excel(workbook=wb,
                                          worksheet="reception",
                                          autofit=True)
        pl.from_pandas(expe).write_excel(workbook=wb,
                                         worksheet="expedition",
                                         autofit=True)
        if not deblocage.empty:
            pl.from_pandas(deblocage).write_excel(workbook=wb,
                                                  worksheet="detail_deblocage",
                                                  autofit=True)
        if not annulation.empty:
            pl.from_pandas(annulation).write_excel(workbook=wb,
                                                  worksheet="detail_annulation",
                                                  autofit=True)
        pl.from_pandas(reception).write_excel(workbook=wb,
                                              worksheet="detail_recep",
                                              autofit=True)
        if not retard.empty:
            pl.from_pandas(retard).write_excel(workbook=wb,
                                               worksheet="retardLivraison",
                                               autofit=True
                                              )
        pl.from_pandas(expedition).write_excel(workbook=wb,
                                               worksheet="detail_expe",
                                               autofit=True)
        pl.from_pandas(achat).write_excel(workbook=wb,
                                          worksheet="commande_achat",
                                          autofit=True)
        pl.from_pandas(vente).write_excel(workbook=wb,
                                          worksheet="commande_vente",
                                          autofit=True)
        pl.from_pandas(prix_achat).write_excel(workbook=wb,
                                               worksheet="prix_achat",
                                               autofit=True)

    return

# Main

In [33]:
# à executer et sera utilisé pour toutes les centra
RefFourCouleur = ref_four_couleur()

for marque in param["Marque"].unique()[0:1]:
    for saison in param[param["Marque"] == marque]["Saison"].unique(): # permet de recupérer les données sur plusieurs saisons
        parametrage = param[(param["Marque"] == marque) & (param["Saison"] == saison)]
        idcat = parametrage["IDCatalogue"].unique()
        print(marque, ":" ,idcat)

        idcat_sql = int_to_sql(idcat)

        # 1- réceptions
        achat = get_achat(idcat_sql)

        idcmd_achat = get_idcmd_achat(achat)
        idcmd_achat_sql = str_to_sql(idcmd_achat)

        reception = get_reception(idcmd_achat_sql)
        
        # 2 - Prix d'achat par RefFour Coul - il faut l'enregistrer dans le fichier excel car besoin ensuite dans le PwBI
        # et dans get_deblocage
        prix_achat = get_PrixAchat_RefFourCoul(achat, RefFourCouleur)

        # 3 - expéditions    
        vente = get_vente(idcat_sql)

        idcmd_vente = get_idcmd_vente(vente)
        idcmd_vente_sql = str_to_sql(idcmd_vente)

        expedition = get_expedition(idcmd_vente_sql)

        # 4 - Totaux commandés pour le calucl des % vs qté & montant commandée dans M3
        qte_achat_tot = achat["QuantiteCommandeePiece"].sum()
        montant_achat_tot = achat["MontantAchat"].sum()

        
        # 5 - déblocages et annulations
        deblocage = get_deblocage(marque, saison, prix_achat)
        annulation = get_annulations(marque, saison, prix_achat)
      
        # 6 - synthese
        synthese = calc_synthese(achat, reception, expedition, deblocage, annulation)
        
        # Compilation des données des "3" df
        deblocage_g, reception_g, expedition_g = aggregate_by_date(deblocage, reception, expedition)
        data = merge_df(deblocage_g, reception_g, expedition_g)
        
        # 7 - agrégation des données en date
        data_by_date = sum_by_date(data)

        #         ### CALCULER LES %###
        
#         # 8 - agrégation par date de déblocage
#         data_by_date_deblocage = sum_by_date_deblocage(data)
        
#         # 9 - calucl des % cumulé vs qté & montant commandée dans M3
#         data_by_date = calcul_pct_cum_df(data_by_date, qte_achat_tot, montant_achat_tot)
#         data_by_date_deblocage = calcul_pct_cum_df(data_by_date_deblocage, qte_achat_tot, montant_achat_tot)
        
#         # 10 - Calcul des % non cumulés vs qté & montant commandée dans M3
#         deblocage_g = calcul_pct_df(deblocage_g, qte_achat_tot, montant_achat_tot)
#         reception_g = calcul_pct_df(reception_g, qte_achat_tot, montant_achat_tot)
#         expedition_g = calcul_pct_df(expedition_g, qte_achat_tot, montant_achat_tot)
        
#         # 11 - calcul du délai de livraison moyen
#         delai_livraison = calc_delai_livraison(deblocage, reception, RefFourCouleur)
        
#         # 12 - calcul retard de livraison
#         retard = calc_retard_livraisons(achat, reception,RefFourCouleur)
                    
#         # ajout des IDCatalogue à réception et expédition
#         reception, expedition = add_idcatalogue(reception, achat, expedition, vente)
        
#         # alertes
#         alertes = get_warnings(deblocage)
        
#         # Enregistrement en polars pour l'autoajustement des colonnes
#         save(marque, saison, parametrage, synthese ,reception , reception_g , expedition ,expedition_g ,
#              achat, vente, deblocage ,deblocage_g , data_by_date, data_by_date_deblocage, prix_achat,
#              delai_livraison, alertes, retard, annulation
#             )
        

synthese

NIKE : [2910 2911 2912 2913 2914 2927 2928 2930 2931 2932 2933]
Nombre de doublons de prix : 0
fichier déblocage : \\BUREAU2022\ACHATSChaussures\LOGISTIQUE\Appros\2. CENTRALISATION NIKE\suivi_deblocage_centra_nike-SU26.xlsx
NIKE : [3002 3006 3032 3033 3035 3036 3037 3038 3039 3040]
Nombre de doublons de prix : 0
fichier déblocage : \\bureau2022\ACHATSChaussures\LOGISTIQUE\Appros\2. CENTRALISATION NIKE\suivi_deblocage_centra_nike-FA26.xlsx


,Unité,Commande,Deblocage,Reception,Expedition,% Deblocage,% Reception,% Expedition,Annulation,% Annulation
0,QuantitePiece,381024.00,156083.00,36411.0,5790.00,0.410,0.096,0.015,7436.00,0.020
1,Valo €,9604111.78,3407928.42,893449.2,170102.26,0.355,0.093,0.018,218499.96,0.023


# Main par ref fournisseur couleur

In [9]:
for marque in param["Marque"].unique()[0:1]:
    print(param[param["Marque"] == marque]["Saison"].unique())

['SU 26' 'FA 26']


In [10]:
param[param["Marque"] == marque]["Saison"].unique()[1:2]

array(['FA 26'], dtype=object)

In [90]:
# à executer et sera utilisé pour toutes les centra
RefFourCouleur = ref_four_couleur()

for marque in param["Marque"].unique()[0:1]:
    for saison in param[param["Marque"] == marque]["Saison"].unique()[1:2]: # permet de recupérer les données sur plusieurs saisons
        parametrage = param[(param["Marque"] == marque) & (param["Saison"] == saison)]
        idcat = parametrage["IDCatalogue"].unique()
        print(marque, ":" ,idcat)

        idcat_sql = int_to_sql(idcat)

        # 1- réceptions
        achat = get_achat(idcat_sql)

        idcmd_achat = get_idcmd_achat(achat)
        idcmd_achat_sql = str_to_sql(idcmd_achat)

        reception = get_reception(idcmd_achat_sql)
        
        # 2 - Prix d'achat par RefFour Coul - il faut l'enregistrer dans le fichier excel car besoin ensuite dans le PwBI
        # et dans get_deblocage
        prix_achat = get_PrixAchat_RefFourCoul(achat, RefFourCouleur)

        # 3 - expéditions    
        vente = get_vente(idcat_sql)

        idcmd_vente = get_idcmd_vente(vente)
        idcmd_vente_sql = str_to_sql(idcmd_vente)

        expedition = get_expedition(idcmd_vente_sql)

        # 4 - Totaux commandés pour le calucl des % vs qté & montant commandée dans M3
        qte_achat_tot = achat["QuantiteCommandeePiece"].sum()
        montant_achat_tot = achat["MontantAchat"].sum()

        
        # 5 - déblocages et annulations
        deblocage = get_deblocage(marque, saison, prix_achat)
        annulation = get_annulations(marque, saison, prix_achat)
      
        # 6 - synthese
        synthese = calc_synthese(achat, reception, expedition, deblocage, annulation)
        
        # 7 - ajouter la ref fournisseur
        achat_ref = groupby_ref_couleur_date(achat, RefFourCouleur) # pour pouvoir récupérer la liste des ref four couleur
        deblocage_ref_date = groupby_ref_couleur_date(deblocage, RefFourCouleur)
        reception_ref_date = groupby_ref_couleur_date(reception, RefFourCouleur)
        expedition_ref_date = groupby_ref_couleur_date(expedition, RefFourCouleur)
        
        # 8 - compiler les données des 3 df
        # pour pouvoir faire des cumsum utilisables dans PowerBi en gardant le détail de la réf couleur,
        # il va falloir une ligne pour chaque date et chaque ReferenceFournisseurCouleur
        ref_date_mvt = get_ref_date(achat_ref, deblocage_ref_date, reception_ref_date, expedition_ref_date)
        
        data =  merge_df_by_ref_four(deblocage_ref_date, reception_ref_date, expedition_ref_date, ref_date_mvt)
        
        data_by_date = cumsum_by_ref_date(data)
        
        # 9 - compiler par date de déblocage
        data_by_date_deblocage = sum_by_ref_date_deblocage(data)
        
        
                        ### CALCULER LES %###
            
        # 10 - calucl des % cumulé vs qté & montant commandée dans M3
        data_by_date = calcul_pct_cum_df(data_by_date, qte_achat_tot, montant_achat_tot)
        data_by_date_deblocage = calcul_pct_cum_df(data_by_date_deblocage, qte_achat_tot, montant_achat_tot)
        
        # 10 - Calcul des % non cumulés vs qté & montant commandée dans M3
        deblocage_ref_date = calcul_pct_df(deblocage_ref_date, qte_achat_tot, montant_achat_tot)
        reception_ref_date = calcul_pct_df(reception_ref_date, qte_achat_tot, montant_achat_tot)
        expedition_ref_date = calcul_pct_df(expedition_ref_date, qte_achat_tot, montant_achat_tot)
        
        # 11 - calcul du délai de livraison moyen
        delai_livraison = calc_delai_livraison_by_ref(deblocage, reception, RefFourCouleur)
        
        # 12 - calcul retard de livraison
        retard = calc_retard_livraisons(achat, reception,RefFourCouleur)
                    
        # ajout des IDCatalogue à réception et expédition
        reception, expedition = add_idcatalogue(reception, achat, expedition, vente)
        
        # alertes
        alertes = get_warnings(deblocage)
        
        # Enregistrement en polars pour l'autoajustement des colonnes
        save_by_ref(marque, saison, parametrage, synthese ,reception , reception_ref_date , expedition ,expedition_ref_date ,
             achat_ref, vente, deblocage ,deblocage_ref_date , data_by_date, data_by_date_deblocage, prix_achat,
             delai_livraison, alertes, retard, annulation
            )

NIKE : [3002 3006 3032 3033 3035 3036 3037 3038 3039 3040]
Nombre de doublons de prix : 0
fichier déblocage : \\bureau2022\ACHATSChaussures\LOGISTIQUE\Appros\2. CENTRALISATION NIKE\suivi_deblocage_centra_nike-FA26.xlsx


In [89]:
def calc_delai_livraison_by_ref(deblocage, reception_ref_date, RefFourCouleur):
    """ le délai théorique entre la livraison et la réception est de 24h pour Logs
        pondéré par la qté receptionnée    
    """
    if not deblocage.empty and not reception.empty:
        # passer les réceptions en par ref fournisseur
        reception_ref = pd.merge(reception, RefFourCouleur[["IDArticle", "ReferenceFournisseurCouleur"]], on="IDArticle", how="left")
        # puis grouper par ref four couleur
        reception_ref = (reception_ref
                         .groupby(["IDCommande", "ReferenceFournisseurCouleur", "DateReception"])[["QuantiteReceptionPiece", "MontantReception"]]
                         .sum()
                         .reset_index()
                        )

        delai = pd.merge(deblocage, reception_ref, on=["IDCommande", "ReferenceFournisseurCouleur"], how="left")
        # supprimer les lignes ou la date de réception est antérieure à la date de déblocage
        delai = delai[delai["DateReception"] > delai["DateDeblocage"]]
        # calculer le delta entre le déblocage et la réception
        delai["delai j"] = delai["DateReception"] - delai["DateDeblocage"] + pd.Timedelta(days=-1)
        # supprimer les lignes où la date de réception est antérieure à la date de déblocage
        delai = delai[delai["delai j"] > timedelta(days=0)]
        
        delai["delai*qte"] = delai["delai j"] * delai["QuantiteReceptionPiece"]/1000 # division par 1000 car sinon renvoi une erreur le chiffre est too big
        
        
        # grouper par date de déblocage - aggreger par moyenne de délai
        delai_g = delai.groupby(["ReferenceFournisseurCouleur","DateDeblocage"], dropna=False).agg({"delai j":"mean", "QuantiteReceptionPiece": sum, "delai*qte":sum}).reset_index()
        # calculer delai moyen pondéré
        delai_g["delai pondere j"] = delai_g["delai*qte"] / delai_g["QuantiteReceptionPiece"] *1000
        delai_g.drop(columns=["QuantiteReceptionPiece", "delai*qte"], inplace=True)
        
    
    else:
        delai_g = pd.DataFrame()
    
    return delai_g

In [88]:
delai_livraison_c = calc_delai_livraison_by_ref(deblocage, reception, RefFourCouleur)
delai_livraison_c

,ReferenceFournisseurCouleur,DateDeblocage,delai j,QuantiteReceptionPiece,delai*qte,delai pondere j
0,394053-005,2026-06-17,8 days 00:00:00,567.0,4 days 12:51:50.400000,8 days 00:00:00
1,394053-010,2026-06-17,7 days 00:00:00,1926.0,13 days 11:34:04.800000,7 days 00:00:00
2,394053-108,2026-06-17,7 days 00:00:00,1161.0,8 days 03:02:52.800000,7 days 00:00:00
3,394055-103,2026-06-17,6 days 00:00:00,2323.0,13 days 22:30:43.200000,6 days 00:00:00
4,AR6029-459,2026-06-17,12 days 00:00:00,1488.0,17 days 20:32:38.400000,12 days 00:00:00
...,...,...,...,...,...,...
61,IR5118-001,2026-06-17,14 days 00:00:00,408.0,5 days 17:05:16.800000,14 days 00:00:00
62,IV5952-100,2026-06-17,8 days 00:00:00,1449.0,11 days 14:12:28.800000,8 days 00:00:00
63,IV5953-100,2026-06-17,11 days 00:00:00,1701.0,18 days 17:03:50.400000,11 days 00:00:00
64,IX0357-437,2026-06-17,9 days 12:00:00,1593.0,16 days 15:10:04.800000,10 days 10:34:34.576271


In [100]:
delai_livraison_c["delai pondere j"].iloc[0].total_seconds()

691200.0

In [101]:
delai_livraison_c["total_seconds"]= delai_livraison_c["delai pondere j"].apply(lambda x: x.total_seconds())

In [104]:
delai_livraison_c["total_seconds"].mean() / (60 *60 *24)

10.980320602143834

In [37]:
delai_livraison_b = calc_delai_livraison(deblocage, reception, RefFourCouleur)
delai_livraison_b

,DateDeblocage,delai j,delai pondere j
0,2026-06-17,11 days 09:06:50.126582278,9 days 21:08:39.982314


In [93]:
delai_livraison_c["delai pondere j"].describe()

count                   66
unique                  10
top       13 days 00:00:00
freq                    22
Name: delai pondere j, dtype: object

In [38]:
delai_livraison

,ReferenceFournisseurCouleur,DateDeblocage,delai j,delai pondere j
0,394053-005,2026-06-17,8 days 00:00:00,8 days 00:00:00
1,394053-010,2026-06-17,7 days 00:00:00,7 days 00:00:00
2,394053-108,2026-06-17,7 days 00:00:00,7 days 00:00:00
3,394055-103,2026-06-17,6 days 00:00:00,6 days 00:00:00
4,AR6029-459,2026-06-17,12 days 00:00:00,12 days 00:00:00
...,...,...,...,...
61,IR5118-001,2026-06-17,14 days 00:00:00,14 days 00:00:00
62,IV5952-100,2026-06-17,8 days 00:00:00,8 days 00:00:00
63,IV5953-100,2026-06-17,11 days 00:00:00,11 days 00:00:00
64,IX0357-437,2026-06-17,9 days 12:00:00,10 days 10:34:34.576271


In [30]:
df = data_by_date_deblocage[data_by_date_deblocage["DateDeblocage"] == data_by_date_deblocage["DateDeblocage"].max()]
df

,ReferenceFournisseurCouleur,DateDeblocage,QuantiteDeblocagePiece,MontantDeblocage,QuantiteReceptionPiece,MontantReception,QuantiteExpeditionPiece,MontantExpedition,QuantiteDeblocagePiece_cum,MontantDeblocage_cum,QuantiteReceptionPiece_cum,MontantReception_cum,QuantiteExpeditionPiece_cum,MontantExpedition_cum,pct_QuantiteDeblocagePiece_cum,pct_QuantiteReceptionPiece_cum,pct_QuantiteExpeditionPiece_cum,pct_MontantDeblocage_cum,pct_MontantReception_cum,pct_MontantExpedition_cum
4,394053-005,2026-07-02,0.0,0.0,567.0,20241.9,333.0,11888.1,567.0,20241.9,567.0,20241.9,333.0,11888.1,0.001501,0.001501,0.000882,0.002133,0.002133,0.001253
9,394053-010,2026-07-02,0.0,0.0,1926.0,68758.2,1179.0,42090.3,2454.0,87607.8,1926.0,68758.2,1179.0,42090.3,0.006496,0.005098,0.003121,0.009232,0.007246,0.004435
14,394053-108,2026-07-02,0.0,0.0,1161.0,41447.7,702.0,25061.4,1551.0,55370.7,1161.0,41447.7,702.0,25061.4,0.004106,0.003073,0.001858,0.005835,0.004368,0.002641
19,394055-004,2026-07-02,0.0,0.0,0.0,0.0,0.0,0.0,603.0,21527.1,0.0,0.0,0.0,0.0,0.001596,0.000000,0.000000,0.002269,0.000000,0.000000
24,394055-103,2026-07-02,0.0,0.0,2323.0,82931.1,1909.0,68151.3,2323.0,82931.1,2323.0,82931.1,1909.0,68151.3,0.006149,0.006149,0.005053,0.008739,0.008739,0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4934,IX0357-437,2026-07-02,0.0,0.0,1593.0,43488.9,450.0,12285.0,1593.0,43488.9,1593.0,43488.9,450.0,12285.0,0.004217,0.004217,0.001191,0.004583,0.004583,0.001295
4939,IX0655-437,2026-07-02,0.0,0.0,0.0,0.0,0.0,0.0,1066.0,24624.6,0.0,0.0,0.0,0.0,0.002822,0.000000,0.000000,0.002595,0.000000,0.000000
4944,IX0702-001,2026-07-02,0.0,0.0,0.0,0.0,0.0,0.0,132.0,5940.0,0.0,0.0,0.0,0.0,0.000349,0.000000,0.000000,0.000626,0.000000,0.000000
4949,IX0702-002,2026-07-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [31]:
df["QuantiteDeblocagePiece_cum"].sum() / qte_achat_tot

0.41318141989998913

In [32]:
df["pct_QuantiteDeblocagePiece_cum"].sum()

0.4131814198999892

In [17]:
qte_achat_tot

377759.0

In [21]:
df["recalc_pct_qte_debl"] = df["QuantiteDeblocagePiece_cum"] / qte_achat_tot

In [22]:
df[["ReferenceFournisseurCouleur", "pct_QuantiteDeblocagePiece_cum", "recalc_pct_qte_debl"]]

,ReferenceFournisseurCouleur,pct_QuantiteDeblocagePiece_cum,recalc_pct_qte_debl
4,394053-005,0.002,0.001501
9,394053-010,0.006,0.006496
14,394053-108,0.004,0.004106
19,394055-004,0.002,0.001596
24,394055-103,0.006,0.006149
...,...,...,...
4934,IX0357-437,0.004,0.004217
4939,IX0655-437,0.003,0.002822
4944,IX0702-001,0.000,0.000349
4949,IX0702-002,0.000,0.000000


In [60]:
data_by_date[data_by_date["ReferenceFournisseurCouleur"] == "394053-010"]

,ReferenceFournisseurCouleur,Date,QuantiteDeblocagePiece,MontantDeblocage,QuantiteReceptionPiece,MontantReception,QuantiteExpeditionPiece,MontantExpedition,QuantiteDeblocagePiece_cum,MontantDeblocage_cum,QuantiteReceptionPiece_cum,MontantReception_cum,QuantiteExpeditionPiece_cum,MontantExpedition_cum,pct_QuantiteDeblocagePiece_cum,pct_QuantiteReceptionPiece_cum,pct_QuantiteExpeditionPiece_cum,pct_MontantDeblocage_cum,pct_MontantReception_cum,pct_MontantExpedition_cum
9,394053-010,2026-06-17,2412.0,86108.4,0.0,0.0,0.0,0.0,2412.0,86108.4,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
10,394053-010,2026-06-19,0.0,0.0,0.0,0.0,0.0,0.0,2412.0,86108.4,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
11,394053-010,2026-06-22,42.0,1499.4,0.0,0.0,0.0,0.0,2454.0,87607.8,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
12,394053-010,2026-06-24,0.0,0.0,0.0,0.0,0.0,0.0,2454.0,87607.8,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
13,394053-010,2026-06-25,0.0,0.0,1926.0,68758.2,0.0,0.0,2454.0,87607.8,1926.0,68758.2,0.0,0.0,0.006,0.005,0.000,0.009,0.007,0.000
14,394053-010,2026-06-26,0.0,0.0,0.0,0.0,0.0,0.0,2454.0,87607.8,1926.0,68758.2,0.0,0.0,0.006,0.005,0.000,0.009,0.007,0.000
15,394053-010,2026-06-29,0.0,0.0,0.0,0.0,54.0,1927.8,2454.0,87607.8,1926.0,68758.2,54.0,1927.8,0.006,0.005,0.000,0.009,0.007,0.000
16,394053-010,2026-06-30,0.0,0.0,0.0,0.0,441.0,15743.7,2454.0,87607.8,1926.0,68758.2,495.0,17671.5,0.006,0.005,0.001,0.009,0.007,0.002
17,394053-010,2026-07-01,0.0,0.0,0.0,0.0,414.0,14779.8,2454.0,87607.8,1926.0,68758.2,909.0,32451.3,0.006,0.005,0.002,0.009,0.007,0.003


In [61]:
data_by_date_deblocage[data_by_date_deblocage["ReferenceFournisseurCouleur"] == "394053-010"]

,ReferenceFournisseurCouleur,DateDeblocage,QuantiteDeblocagePiece,MontantDeblocage,QuantiteReceptionPiece,MontantReception,QuantiteExpeditionPiece,MontantExpedition,QuantiteDeblocagePiece_cum,MontantDeblocage_cum,QuantiteReceptionPiece_cum,MontantReception_cum,QuantiteExpeditionPiece_cum,MontantExpedition_cum,pct_QuantiteDeblocagePiece_cum,pct_QuantiteReceptionPiece_cum,pct_QuantiteExpeditionPiece_cum,pct_MontantDeblocage_cum,pct_MontantReception_cum,pct_MontantExpedition_cum
5,394053-010,2026-06-17,2412.0,86108.4,0.0,0.0,0.0,0.0,2412.0,86108.4,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
6,394053-010,2026-06-19,0.0,0.0,0.0,0.0,0.0,0.0,2412.0,86108.4,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
7,394053-010,2026-06-22,42.0,1499.4,0.0,0.0,0.0,0.0,2454.0,87607.8,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
8,394053-010,2026-06-24,0.0,0.0,0.0,0.0,0.0,0.0,2454.0,87607.8,0.0,0.0,0.0,0.0,0.006,0.000,0.000,0.009,0.000,0.000
9,394053-010,2026-07-01,0.0,0.0,1926.0,68758.2,909.0,32451.3,2454.0,87607.8,1926.0,68758.2,909.0,32451.3,0.006,0.005,0.002,0.009,0.007,0.003


In [239]:
data_by_date[data_by_date["ReferenceFournisseurCouleur"] == "FN3861-051"]

,ReferenceFournisseurCouleur,Date,QuantiteDeblocagePiece,MontantDeblocage,QuantiteReceptionPiece,MontantReception,QuantiteExpeditionPiece,MontantExpedition,QuantiteDeblocagePiece_cum,MontantDeblocage_cum,QuantiteReceptionPiece_cum,MontantReception_cum,QuantiteExpeditionPiece_cum,MontantExpedition_cum
1392,FN3861-051,2026-06-17,2303.0,67708.2,0.0,0.0,0.0,0.0,2303.0,67708.2,0.0,0.0,0.0,0.0
1393,FN3861-051,2026-06-19,0.0,0.0,0.0,0.0,0.0,0.0,2303.0,67708.2,0.0,0.0,0.0,0.0
1394,FN3861-051,2026-06-22,0.0,0.0,0.0,0.0,0.0,0.0,2303.0,67708.2,0.0,0.0,0.0,0.0
1395,FN3861-051,2026-06-24,0.0,0.0,0.0,0.0,9.0,264.6,2303.0,67708.2,0.0,0.0,9.0,264.6
1396,FN3861-051,2026-06-25,0.0,0.0,0.0,0.0,0.0,0.0,2303.0,67708.2,0.0,0.0,9.0,264.6
1397,FN3861-051,2026-06-26,0.0,0.0,287.0,8437.8,0.0,0.0,2303.0,67708.2,287.0,8437.8,9.0,264.6
1398,FN3861-051,2026-06-29,0.0,0.0,0.0,0.0,18.0,529.2,2303.0,67708.2,287.0,8437.8,27.0,793.8
1399,FN3861-051,2026-06-30,0.0,0.0,0.0,0.0,31.0,911.4,2303.0,67708.2,287.0,8437.8,58.0,1705.2


In [18]:
data[data["ReferenceFournisseurCouleur"] == "394053-010"]

,ReferenceFournisseurCouleur,Date,DateDeblocage,QuantiteDeblocagePiece,MontantDeblocage,DateReception,QuantiteReceptionPiece,MontantReception,DateExpedition,QuantiteExpeditionPiece,MontantExpedition
9,394053-010,2026-06-17,2026-06-17,2412.0,86108.4,NaN,NaN,NaN,NaN,NaN,NaN
10,394053-010,2026-06-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,394053-010,2026-06-22,2026-06-22,42.0,1499.4,NaN,NaN,NaN,NaN,NaN,NaN
12,394053-010,2026-06-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,394053-010,2026-06-25,NaN,NaN,NaN,2026-06-25,1926.0,68758.2,NaN,NaN,NaN
14,394053-010,2026-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,394053-010,2026-06-29,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-29,54.0,1927.8
16,394053-010,2026-06-30,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-30,441.0,15743.7
17,394053-010,2026-07-01,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01,414.0,14779.8


In [62]:
qte_achat_tot

377759.0

In [81]:
annulation

,IDCommande,ReferenceFournisseurCouleur,QuantiteAnnulationPiece
0,2049127,II9943-010,612
1,2048475,HQ4335-907,108
2,2053756,II1965-417,1224
3,2048524,IB4378-006,84
4,2048991,HQ6413-100,1143
5,2048991,IR2949-100,1143
6,2048482,IR2949-100,171
7,2048514,IB1530-019,96
8,2048296,DH6613-100,168
9,2048429,FZ7626-394,42


In [66]:
retard = retard.reset_index()
retard["RetardMoyenPondéré"] = retard["ReceptionQteDiv1000"] / retard["QuantiteReceptionPiece"] * 1000
retard

,index,ReferenceFournisseurCouleur,IDCommandeAchat,DateLivraisonAttendue,DateReceptionMin,DateReceptionMax,ReceptionQteDiv1000,QuantiteReceptionPiece,RetardMoyenPondéré
0,0,394053-005,2048795,2026-03-20,2026-05-07,2026-05-07,26 days 15:34:33.600000,567.0,47 days
1,1,394053-100,2048796,2026-03-20,2026-04-13,2026-04-13,15 days 01:00:28.800000,654.0,23 days
2,2,394053-100,2048797,2026-03-20,2026-04-13,2026-04-13,51 days 18:00:00,2250.0,23 days
3,3,394053-103,2048798,2026-03-20,2026-04-22,2026-04-22,18 days 17:16:48,585.0,32 days
4,4,394055-100,2048165,2026-03-20,2026-04-21,2026-04-21,27 days 00:01:26.400000,871.0,31 days
...,...,...,...,...,...,...,...,...,...
856,856,IR5202-100,2049174,2026-03-20,2026-04-08,2026-04-08,7 days 22:30:43.200000,441.0,18 days
857,857,IR5202-101,2048793,2026-03-20,2026-05-18,2026-05-18,1 days 09:24:28.800000,24.0,58 days
858,858,IR5202-101,2049175,2026-03-20,2026-04-24,2026-04-24,17 days 03:15:50.400000,504.0,34 days
859,859,IR7888-133,2048794,2026-03-20,2026-04-29,2026-04-29,2 days 00:40:19.200000,52.0,39 days


In [15]:
achat

,IDCommandeAchat,IDCatalogue,IDArticle,DateLivraisonAttendue,QuantiteCommandeePiece,PrixAchatPiece,MontantAchat
0,2048589,2927,88015-0102,2026-03-20,32.0,33.6,1075.2
1,2048146,2930,86357-0303,2026-03-20,2.0,71.4,142.8
2,2048266,2927,63876-5206,2026-03-20,68.0,10.5,714.0
3,2048159,2931,88236-0104,2026-05-20,7.0,126.0,882.0
4,2048621,2927,88035-0103,2026-05-20,45.0,21.0,945.0
...,...,...,...,...,...,...,...
3654,2048986,2927,83272PCK17,2026-03-20,1485.0,29.4,43659.0
3655,2048630,2927,86272-0202,2026-03-20,9.0,21.0,189.0
3656,2048645,2927,88047-0204,2026-04-20,46.0,10.5,483.0
3657,2048648,2927,88048-0202,2026-03-20,14.0,18.9,264.6


In [16]:
reception

,IDCommande,IDArticle,DateReception,QuantiteReceptionPiece,MontantReception,IDCatalogue,IDCommandeAchat
0,2047972,71245-0101,2026-04-20,4.0,140.0,2914,2047972
1,2047972,71245-0102,2026-04-20,6.0,210.0,2914,2047972
2,2047972,71245-0103,2026-04-20,4.0,140.0,2914,2047972
3,2047972,71245-0104,2026-04-20,2.0,70.0,2914,2047972
4,2047973,71250-0101,2026-04-23,6.0,315.0,2914,2047973
...,...,...,...,...,...,...,...
2692,2049233,88261-0107,2026-04-28,16.0,873.6,2933,2049233
2693,2049233,88261-0108,2026-04-28,13.0,709.8,2933,2049233
2694,2049233,88261-0109,2026-04-28,6.0,327.6,2933,2049233
2695,2049233,88261-0110,2026-04-28,6.0,327.6,2933,2049233


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3675 entries, 0 to 3674
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   IDCommandeAchat_x       3675 non-null   object 
 1   IDCatalogue_x           3675 non-null   int64  
 2   IDArticle               3675 non-null   object 
 3   DateLivraisonAttendue   3675 non-null   object 
 4   QuantiteCommandeePiece  3675 non-null   float64
 5   PrixAchatPiece          3675 non-null   float64
 6   MontantAchat            3675 non-null   float64
 7   IDCommande              2697 non-null   object 
 8   DateReception           2697 non-null   object 
 9   QuantiteReceptionPiece  2697 non-null   float64
 10  MontantReception        2697 non-null   float64
 11  IDCatalogue_y           2697 non-null   float64
 12  IDCommandeAchat_y       2697 non-null   object 
dtypes: float64(6), int64(1), object(6)
memory usage: 373.4+ KB


In [20]:
reception[reception[["IDCommande", "IDArticle"]].duplicated()]

,IDCommande,IDArticle,DateReception,QuantiteReceptionPiece,MontantReception,IDCatalogue,IDCommandeAchat
293,2048107,88191PCK01,2026-05-18,1974.0,78960.0,2928,2048107
305,2048112,88200-0101,2026-05-11,335.0,16750.0,2928,2048112
306,2048112,88200-0101,2026-05-12,4.0,200.0,2928,2048112
309,2048112,88200-0103,2026-05-11,98.0,4900.0,2928,2048112
310,2048112,88200-0103,2026-05-12,2.0,100.0,2928,2048112
313,2048113,88200-0102,2026-05-27,720.0,36000.0,2928,2048113
317,2048114,88201-0103,2026-05-18,8.0,400.0,2928,2048114
522,2048166,76740PCK10,2026-03-19,4056.0,142771.2,2932,2048166
1196,2048454,80666-1303,2026-06-02,6.0,214.2,2927,2048454
2348,2048924,78187PCK06,2026-04-15,150.0,2520.0,2927,2048924


In [21]:
reception[(reception["IDCommande"] =="2048107") & (reception["IDArticle"] =="88191PCK01")]

,IDCommande,IDArticle,DateReception,QuantiteReceptionPiece,MontantReception,IDCatalogue,IDCommandeAchat
292,2048107,88191PCK01,2026-05-13,2905.0,116200.0,2928,2048107
293,2048107,88191PCK01,2026-05-18,1974.0,78960.0,2928,2048107


In [22]:
achat[(achat["IDCommandeAchat"] =="2048107") & (achat["IDArticle"] =="88191PCK01")]

,IDCommandeAchat,IDCatalogue,IDArticle,DateLivraisonAttendue,QuantiteCommandeePiece,PrixAchatPiece,MontantAchat
2974,2048107,2928,88191PCK01,2026-03-20,4900.0,40.0,196000.0


In [28]:
df[(df["IDCommandeAchat_x"] =="2048107") & (df["IDArticle"] =="88191PCK01")]

,IDCommandeAchat_x,IDCatalogue_x,IDArticle,DateLivraisonAttendue,QuantiteCommandeePiece,PrixAchatPiece,MontantAchat,IDCommande,DateReception,QuantiteReceptionPiece,MontantReception,IDCatalogue_y,IDCommandeAchat_y,Reception-attendu
2986,2048107,2928,88191PCK01,2026-03-20,4900.0,40.0,196000.0,2048107,2026-05-13,2905.0,116200.0,2928.0,2048107,53 days
2987,2048107,2928,88191PCK01,2026-03-20,4900.0,40.0,196000.0,2048107,2026-05-18,1974.0,78960.0,2928.0,2048107,58 days


In [29]:
df["Reception-attendu"].min()

Timedelta('-14 days +00:00:00')

In [ ]:
"DateDeblocage", "QuantiteDeblocagePiece", "MontantDeblocage", "QuantiteReceptionPiece", "MontantReception", "QuantiteExpeditionPiece", "MontantExpedition", "QuantiteDeblocagePiece_cum", "MontantDeblocage_cum", "QuantiteReceptionPiece_cum", "MontantReception_cum", "QuantiteExpeditionPiece_cum", "MontantExpedition_cum", "pct_QuantiteDeblocagePiece_cum", "pct_QuantiteReceptionPiece_cum", "pct_QuantiteExpeditionPiece_cum", "pct_MontantDeblocage_cum", "pct_MontantReception_cum", "pct_MontantExpedition_cum"


######################################################################################################################